# BTS Flight Departures Scraper (2024)
Scrapes all U.S. airport departure data for 2024 from the BTS On-Time Departures page.

## 1. Imports & Setup

In [4]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.alert import Alert
from selenium.webdriver.support.ui import Select, WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
import pandas as pd
import time
from io import StringIO
import os
import glob

URL = "https://www.transtats.bts.gov/ontime/Departures.aspx"

## 2. Launch Browser & Get All Airports

In [6]:
DOWNLOAD_DIR = "/Users/lolo/Desktop/Classes/winter'26/PIC16B/Projects/Flight-Delays-Prediction-Model"

def create_driver():
    options = webdriver.ChromeOptions()
    options.add_experimental_option("prefs", {
        "download.default_directory": DOWNLOAD_DIR,
        "download.prompt_for_download": False,
    })
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
    wait = WebDriverWait(driver, 15)
    return driver, wait

driver, wait = create_driver()
driver.get(URL)

## 3. Scrape All Airports (2024)

In [8]:
top_airlines = [
    "American Airlines Inc. (AA)",
    "Delta Airlines Inc. (DL)",
    "Southwest Airlines Co. (WN)",
    "United Airlines Inc. (UA)",
]

top_airports = [
    "Newark, NJ: Newark Liberty International (EWR)",
    "San Francisco, CA: San Francisco International (SFO)",
    "Chicago, IL: Chicago O'Hare International (ORD)",
    "New York, NY: LaGuardia (LGA)",
    "New York, NY: John F. Kennedy International (JFK)",
    "Boston, MA: Logan International (BOS)",
    "Denver, CO: Denver International (DEN)",
    "Dallas/Fort Worth, TX: Dallas/Fort Worth International (DFW)",
    "Houston, TX: George Bush Intercontinental/Houston (IAH)",
    "Philadelphia, PA: Philadelphia International (PHL)",
    "Miami, FL: Miami International (MIA)",
    "Washington, DC: Ronald Reagan Washington National (DCA)",
    "Washington, DC: Washington Dulles International (IAD)",
    "Orlando, FL: Orlando International (MCO)",
    "Atlanta, GA: Hartsfield-Jackson Atlanta International (ATL)",
    "Los Angeles, CA: Los Angeles International (LAX)",
    "Seattle, WA: Seattle/Tacoma International (SEA)",
    "Phoenix, AZ: Phoenix Sky Harbor International (PHX)",
    "Minneapolis, MN: Minneapolis-St Paul International (MSP)",
    "Detroit, MI: Detroit Metro Wayne County (DTW)",
    "Charlotte, NC: Charlotte Douglas International (CLT)",
    "Las Vegas, NV: Harry Reid International (LAS)",
    "Baltimore, MD: Baltimore/Washington International Thurgood Marshall (BWI)",
    "Salt Lake City, UT: Salt Lake City International (SLC)",
    "Portland, OR: Portland International (PDX)",
    "San Diego, CA: San Diego International (SAN)",
    "Tampa, FL: Tampa International (TPA)",
    "Nashville, TN: Nashville International (BNA)",
    "Austin, TX: Austin - Bergstrom International (AUS)",
    "Houston, TX: William P Hobby (HOU)",
    "Chicago, IL: Chicago Midway International (MDW)",
    "New Orleans, LA: Louis Armstrong New Orleans International (MSY)",
    "Pittsburgh, PA: Pittsburgh International (PIT)",
    "Cleveland, OH: Cleveland-Hopkins International (CLE)",
    "Buffalo, NY: Buffalo Niagara International (BUF)",
    "Albany, NY: Albany International (ALB)",
    "Syracuse, NY: Syracuse Hancock International (SYR)",
    "Rochester, NY: Frederick Douglass Grtr Rochester International (ROC)",
    "Hartford, CT: Bradley International (BDL)",
    "Providence, RI: Rhode Island Tf Green International (PVD)",
    "Portland, ME: Portland International Jetport (PWM)",
    "Burlington, VT: Burlington International (BTV)",
    "Cincinnati, OH: Cincinnati/Northern Kentucky International (CVG)",
    "Columbus, OH: John Glenn Columbus International (CMH)",
    "Indianapolis, IN: Indianapolis International (IND)",
    "Milwaukee, WI: General Mitchell International (MKE)",
    "Kansas City, MO: Kansas City International (MCI)",
    "St. Louis, MO: St Louis Lambert International (STL)",
    "Memphis, TN: Memphis International (MEM)",
    "Birmingham, AL: Birmingham-Shuttlesworth International (BHM)",
    "Raleigh/Durham, NC: Raleigh-Durham International (RDU)",
    "Richmond, VA: Richmond International (RIC)",
    "Norfolk, VA: Norfolk International (ORF)",
    "Jacksonville, FL: Jacksonville International (JAX)",
    "Fort Lauderdale, FL: Fort Lauderdale-Hollywood International (FLL)",
    "West Palm Beach/Palm Beach, FL: Palm Beach International (PBI)",
    "Sarasota/Bradenton, FL: Sarasota/Bradenton International (SRQ)",
    "Fort Myers, FL: Southwest Florida International (RSW)",
    "San Antonio, TX: San Antonio International (SAT)",
    "Dallas, TX: Dallas Love Field (DAL)",
    "El Paso, TX: El Paso International (ELP)",
    "Albuquerque, NM: Albuquerque International Sunport (ABQ)",
    "Tucson, AZ: Tucson International (TUS)",
    "Colorado Springs, CO: City of Colorado Springs Municipal (COS)",
    "Boise, ID: Boise Air Terminal (BOI)",
    "Spokane, WA: Spokane International (GEG)",
    "Anchorage, AK: Ted Stevens Anchorage International (ANC)",
    "Honolulu, HI: Daniel K Inouye International (HNL)",
    "Kahului, HI: Kahului Airport (OGG)",
    "Oakland, CA: Metro Oakland International (OAK)",
    "San Jose, CA: Norman Y. Mineta San Jose International (SJC)",
    "Sacramento, CA: Sacramento International (SMF)",
    "Long Beach, CA: Long Beach Airport (LGB)",
    "Santa Ana, CA: John Wayne Airport-Orange County (SNA)",
    "Burbank, CA: Bob Hope (BUR)",
    "Reno, NV: Reno/Tahoe International (RNO)",
    "Ontario, CA: Ontario International (ONT)",
    "Omaha, NE: Eppley Airfield (OMA)",
    "Des Moines, IA: Des Moines International (DSM)",
    "Grand Rapids, MI: Gerald R. Ford International (GRR)",
    "Fargo, ND: Hector International (FAR)",
    "Bismarck/Mandan, ND: Bismarck Municipal (BIS)",
    "Billings, MT: Billings Logan International (BIL)",
    "Bozeman, MT: Bozeman Yellowstone International (BZN)",
    "Kalispell, MT: Glacier Park International (FCA)",
    "Jackson, WY: Jackson Hole (JAC)",
    "Aspen, CO: Aspen Pitkin County Sardy Field (ASE)",
    "Montrose/Delta, CO: Montrose Regional (MTJ)",
    "Knoxville, TN: McGhee Tyson (TYS)",
    "Louisville, KY: Louisville Muhammad Ali International (SDF)",
    "Lexington, KY: Blue Grass (LEX)",
    "Greer, SC: Greenville-Spartanburg International (GSP)",
    "Oklahoma City, OK: Okc Will Rogers International (OKC)",
    "Tulsa, OK: Tulsa International (TUL)",
    "Wichita, KS: Wichita Dwight D Eisenhower National (ICT)",
]

all_data = []

In [9]:
all_data = []

for airport in top_airports:
    for airline in top_airlines:
        print(f"Scraping: {airport} | {airline}")
        retries = 3
        while retries > 0:
            try:
                driver.get(URL)
                time.sleep(2)
                airport_dd = Select(wait.until(EC.presence_of_element_located((By.NAME, "cboAirport"))))
                airport_dd.select_by_visible_text(airport)
                time.sleep(1)
                airline_dd = Select(driver.find_element(By.NAME, "cboAirline"))
                airline_dd.select_by_visible_text(airline)
                all_stats_cb = driver.find_element(By.NAME, "chkAllStatistics")
                if not all_stats_cb.is_selected():
                    all_stats_cb.click()
                all_months_cb = driver.find_element(By.NAME, "chkAllMonths")
                if not all_months_cb.is_selected():
                    all_months_cb.click()
                all_days_cb = driver.find_element(By.NAME, "chkAllDays")
                if not all_days_cb.is_selected():
                    all_days_cb.click()
                all_years_cb = driver.find_element(By.NAME, "chkAllYears")
                if all_years_cb.is_selected():
                    all_years_cb.click()
                year_cb = driver.find_element(By.XPATH, "//input[@name='chkYears$37']")
                if not year_cb.is_selected():
                    year_cb.click()
                driver.find_element(By.XPATH, "//input[@value='Submit']").click()
                time.sleep(4)
                try:
                    csv_link = driver.find_element(By.XPATH, "//a[contains(text(), 'CSV')]")
                except:
                    print(f"  No data, skipping")
                    break
                for f in glob.glob(os.path.join(DOWNLOAD_DIR, "*.csv")):
                    os.remove(f)
                csv_link.click()
                downloaded_file = None
                for _ in range(30):
                    time.sleep(1)
                    csv_files = glob.glob(os.path.join(DOWNLOAD_DIR, "*.csv"))
                    if csv_files:
                        downloaded_file = csv_files[0]
                        break
                if not downloaded_file:
                    print(f"  Download timed out, skipping")
                    break
                df = pd.read_csv(downloaded_file, skiprows=7)
                df["airport"] = airport
                df["airline"] = airline
                all_data.append(df)
                print(f"  Got {len(df)} rows | Total so far: {sum(len(d) for d in all_data)}")
                break

            except Exception as e:
                retries -= 1
                print(f"  Error: {e} | Retries left: {retries}")
                try:
                    Alert(driver).accept()
                except:
                    pass
                try:
                    driver.quit()
                except:
                    pass
                time.sleep(5)
                driver, wait = create_driver()

        if len(all_data) > 0 and len(all_data) % 5 == 0:
            pd.concat(all_data, ignore_index=True).to_csv("departures_2024_partial.csv", index=False)
            print("  Progress saved!")

if all_data:
    final_df = pd.concat(all_data, ignore_index=True)
    final_df.to_csv("departures_2024_all.csv", index=False)
    print(f"Done! Total rows: {len(final_df)}")
else:
    print("No data collected.")

Scraping: Newark, NJ: Newark Liberty International (EWR) | American Airlines Inc. (AA)
  Error: Message: 
Stacktrace:
0   chromedriver                        0x00000001010696b4 cxxbridge1$str$ptr + 3127600
1   chromedriver                        0x0000000101061a50 cxxbridge1$str$ptr + 3095756
2   chromedriver                        0x0000000100b3e56c _RNvCsdExgN8vFLbb_7___rustc35___rust_no_alloc_shim_is_unstable_v2 + 75432
3   chromedriver                        0x0000000100b87864 _RNvCsdExgN8vFLbb_7___rustc35___rust_no_alloc_shim_is_unstable_v2 + 375200
4   chromedriver                        0x0000000100bc6620 _RNvCsdExgN8vFLbb_7___rustc35___rust_no_alloc_shim_is_unstable_v2 + 632668
5   chromedriver                        0x0000000100b7bb9c _RNvCsdExgN8vFLbb_7___rustc35___rust_no_alloc_shim_is_unstable_v2 + 326872
6   chromedriver                        0x0000000101028680 cxxbridge1$str$ptr + 2861308
7   chromedriver                        0x000000010102bdd4 cxxbridge1$str$ptr + 287

## 4. Save Final CSV & Close Browser

In [11]:
final_df = pd.read_csv(DOWNLOAD_DIR + "/departures_2024_all.csv")
final_df

,Carrier Code,Date (MM/DD/YYYY),Flight Number,Tail Number,Destination Airport,Scheduled departure time,Actual departure time,Scheduled elapsed time (Minutes),Actual elapsed time (Minutes),Departure delay (Minutes),Wheels-off time,Taxi-Out time (Minutes),Delay Carrier (Minutes),Delay Weather (Minutes),Delay National Aviation System (Minutes),Delay Security (Minutes),Delay Late Aircraft Arrival (Minutes),airport,airline
0,DL,01/01/2024,466.0,N943AT,DTW,13:28,13:40,110.0,92.0,12.0,13:52,12.0,0.0,0.0,0.0,0.0,0.0,"Newark, NJ: Newark Liberty International (EWR)",Delta Airlines Inc. (DL)
1,DL,01/01/2024,917.0,N310DU,SLC,07:00,06:55,325.0,291.0,-5.0,07:18,23.0,0.0,0.0,0.0,0.0,0.0,"Newark, NJ: Newark Liberty International (EWR)",Delta Airlines Inc. (DL)
2,DL,01/01/2024,1064.0,N967AT,MSP,12:39,12:29,184.0,152.0,-10.0,12:42,13.0,0.0,0.0,0.0,0.0,0.0,"Newark, NJ: Newark Liberty International (EWR)",Delta Airlines Inc. (DL)
3,DL,01/01/2024,1067.0,N995AT,MSP,18:29,18:33,194.0,161.0,4.0,18:53,20.0,0.0,0.0,0.0,0.0,0.0,"Newark, NJ: Newark Liberty International (EWR)",Delta Airlines Inc. (DL)
4,DL,01/01/2024,1272.0,N319DU,SLC,18:43,19:01,330.0,278.0,18.0,19:18,17.0,0.0,0.0,0.0,0.0,0.0,"Newark, NJ: Newark Liberty International (EWR)",Delta Airlines Inc. (DL)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3967204,UA,12/30/2024,786.0,N24729,ORD,07:30,10:11,120.0,101.0,161.0,10:20,9.0,142.0,0.0,0.0,0.0,0.0,"Wichita, KS: Wichita Dwight D Eisenhower Natio...",United Airlines Inc. (UA)
3967205,UA,12/30/2024,2354.0,N838UA,DEN,09:25,09:23,105.0,152.0,-2.0,10:34,71.0,0.0,0.0,45.0,0.0,0.0,"Wichita, KS: Wichita Dwight D Eisenhower Natio...",United Airlines Inc. (UA)
3967206,UA,12/31/2024,786.0,N24729,ORD,07:30,07:28,120.0,128.0,-2.0,07:43,15.0,0.0,0.0,0.0,0.0,0.0,"Wichita, KS: Wichita Dwight D Eisenhower Natio...",United Airlines Inc. (UA)
3967207,UA,12/31/2024,2354.0,N890UA,DEN,09:25,09:17,105.0,112.0,-8.0,09:33,16.0,0.0,0.0,0.0,0.0,0.0,"Wichita, KS: Wichita Dwight D Eisenhower Natio...",United Airlines Inc. (UA)


In [12]:
final_df[final_df['Delay Weather (Minutes)'] != 0]

,Carrier Code,Date (MM/DD/YYYY),Flight Number,Tail Number,Destination Airport,Scheduled departure time,Actual departure time,Scheduled elapsed time (Minutes),Actual elapsed time (Minutes),Departure delay (Minutes),Wheels-off time,Taxi-Out time (Minutes),Delay Carrier (Minutes),Delay Weather (Minutes),Delay National Aviation System (Minutes),Delay Security (Minutes),Delay Late Aircraft Arrival (Minutes),airport,airline
125,DL,01/09/2024,2322.0,N943AT,ATL,18:27,19:03,156.0,182.0,36.0,19:29,26.0,4.0,22.0,26.0,0.0,10.0,"Newark, NJ: Newark Liberty International (EWR)",Delta Airlines Inc. (DL)
130,DL,01/09/2024,2434.0,N928AT,ATL,14:36,15:19,145.0,146.0,43.0,15:37,18.0,0.0,7.0,1.0,0.0,36.0,"Newark, NJ: Newark Liberty International (EWR)",Delta Airlines Inc. (DL)
201,DL,01/13/2024,2596.0,N892AT,DTW,06:00,10:01,115.0,123.0,241.0,10:22,21.0,0.0,241.0,8.0,0.0,0.0,"Newark, NJ: Newark Liberty International (EWR)",Delta Airlines Inc. (DL)
238,DL,01/16/2024,791.0,N311DU,SLC,16:55,19:12,320.0,314.0,137.0,19:30,18.0,0.0,18.0,0.0,0.0,113.0,"Newark, NJ: Newark Liberty International (EWR)",Delta Airlines Inc. (DL)
242,DL,01/16/2024,2171.0,N820DN,ATL,15:59,18:10,145.0,176.0,131.0,18:50,40.0,0.0,9.0,31.0,0.0,122.0,"Newark, NJ: Newark Liberty International (EWR)",Delta Airlines Inc. (DL)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3966947,UA,08/23/2024,2091.0,N892UA,ORD,07:35,07:49,113.0,122.0,14.0,08:04,15.0,0.0,14.0,9.0,0.0,0.0,"Wichita, KS: Wichita Dwight D Eisenhower Natio...",United Airlines Inc. (UA)
3966952,UA,08/25/2024,2438.0,N421UA,DEN,09:50,11:27,90.0,83.0,97.0,11:41,14.0,0.0,90.0,0.0,0.0,0.0,"Wichita, KS: Wichita Dwight D Eisenhower Natio...",United Airlines Inc. (UA)
3967064,UA,10/20/2024,2380.0,N427UA,DEN,09:30,10:34,91.0,100.0,64.0,11:01,27.0,0.0,64.0,9.0,0.0,0.0,"Wichita, KS: Wichita Dwight D Eisenhower Natio...",United Airlines Inc. (UA)
3967151,UA,12/03/2024,786.0,N893UA,ORD,07:40,08:06,114.0,119.0,26.0,08:15,9.0,0.0,26.0,5.0,0.0,0.0,"Wichita, KS: Wichita Dwight D Eisenhower Natio...",United Airlines Inc. (UA)
